# 스마트 창고 출고 지연 예측 AI 경진대회

**플랫폼**: DACON  
**기간**: 2026.04.01 ~ 2026.05.04  
**참가자**: 1,006명  
**평가 지표**: MAE (Mean Absolute Error)

## 1. 문제 정의

### 목표
스마트 창고의 현재 운영 상태를 나타내는 센서 데이터와 운영 지표를 기반으로  
**향후 30분간 평균 출고 지연 시간(분)** 을 예측한다.

### 구조
- 하나의 **시나리오(scenario_id)** = 하나의 창고 운영 시뮬레이션 세션  
- 각 시나리오는 **25개의 타임슬롯**으로 구성 (timeslot_idx: 0 ~ 24)  
- 각 타임슬롯은 15분 단위의 스냅샷  
- 예측 대상: 해당 타임슬롯 기준 **다음 30분간 평균 출고 지연 시간**

### 평가 지표
$$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

낮을수록 좋음. 지연 시간(분) 단위이므로 MAE 1.0 = 평균 1분 오차.

## 2. 데이터 파일 구성

| 파일 | 설명 |
|------|------|
| `train.csv` | 학습 데이터 (피처 + 타깃) |
| `test.csv` | 예측 대상 데이터 (타깃 없음) |
| `layout_info.csv` | 창고 레이아웃 메타 정보 |
| `sample_submission.csv` | 제출 양식 |

## 3. train.csv 컬럼 정의

### 3-1. 식별자 컬럼

| 컬럼 | 설명 |
|------|------|
| `ID` | 샘플 고유 식별자 |
| `layout_id` | 창고 레이아웃 ID (layout_info.csv와 조인 키) |
| `scenario_id` | 시나리오 ID (25개 타임슬롯 묶음 단위) |

### 3-2. 주문·처리 피처 (Order & Processing)

| 컬럼 | 설명 |
|------|------|
| `order_inflow_15m` | 15분간 유입된 주문 수 |
| `unique_sku_15m` | 15분간 처리된 고유 SKU(재고 품목) 수 |
| `avg_items_per_order` | 주문당 평균 아이템 수 |
| `urgent_order_ratio` | 긴급 주문 비율 (0~1) |
| `heavy_item_ratio` | 중량 아이템 비율 (0~1) |
| `cold_chain_ratio` | 냉장 체인 필요 아이템 비율 (0~1) |
| `sku_concentration` | SKU 집중도 (특정 SKU에 주문이 몰리는 정도) |
| `order_wave_count` | 주문 웨이브(배치 처리 단위) 수 |
| `pick_list_length_avg` | 평균 피킹 리스트 길이 |
| `express_lane_util` | 익스프레스 레인 활용률 |
| `bulk_order_ratio` | 대량 주문 비율 |
| `return_order_ratio` | 반품 주문 비율 |
| `prev_shift_volume` | 이전 교대 처리 물량 |
| `daily_forecast_accuracy` | 당일 수요 예측 정확도 |

### 3-3. 로봇·AGV 피처 (Robot & AGV)

| 컬럼 | 설명 |
|------|------|
| `robot_active` | 현재 활성(작업 중) 로봇 수 |
| `robot_idle` | 현재 대기 중 로봇 수 |
| `robot_charging` | 현재 충전 중 로봇 수 |
| `robot_utilization` | 로봇 가동률 (active / total) |
| `avg_trip_distance` | 평균 이동 거리 |
| `task_reassign_15m` | 15분간 태스크 재할당(경로 변경) 횟수 |
| `battery_mean` | 로봇 배터리 평균 잔량 |
| `battery_std` | 로봇 배터리 표준편차 |
| `low_battery_ratio` | 저배터리 로봇 비율 |
| `charge_queue_length` | 충전 대기열 길이 |
| `avg_charge_wait` | 평균 충전 대기 시간 |
| `charge_efficiency_pct` | 충전 효율 (%) |
| `battery_cycle_count_avg` | 평균 배터리 사이클 수 |
| `agv_task_success_rate` | AGV 태스크 성공률 |
| `robot_calibration_score` | 로봇 캘리브레이션(정밀도) 점수 |
| `robot_firmware_update_days` | 마지막 펌웨어 업데이트 이후 경과 일수 |
| `avg_idle_duration_min` | 평균 유휴 시간(분) |
| `fleet_age_months_avg` | 로봇 차량 평균 운용 개월 수 |

### 3-4. 혼잡·장애 피처 (Congestion & Fault)

| 컬럼 | 설명 |
|------|------|
| `congestion_score` | 전체 혼잡도 점수 |
| `max_zone_density` | 최대 구역 밀도 (가장 혼잡한 구역) |
| `blocked_path_15m` | 15분간 차단된 경로 수 |
| `near_collision_15m` | 15분간 충돌 위험 감지 횟수 |
| `fault_count_15m` | 15분간 장애(오류) 발생 횟수 |
| `avg_recovery_time` | 장애 평균 복구 시간 |
| `aisle_traffic_score` | 통로 교통 흐름 점수 |
| `intersection_wait_time_avg` | 평균 교차로 대기 시간 |
| `path_optimization_score` | 경로 최적화 점수 |

### 3-5. 패킹·출하 피처 (Packing & Outbound)

| 컬럼 | 설명 |
|------|------|
| `pack_utilization` | 패킹 스테이션 활용률 |
| `manual_override_ratio` | 수동 개입(자동화 우선순위 변경) 비율 |
| `replenishment_overlap` | 보충 작업과 출고 작업의 중복 정도 |
| `loading_dock_util` | 로딩 독 활용률 |
| `staging_area_util` | 스테이징 영역 활용률 |
| `outbound_truck_wait_min` | 아웃바운드 트럭 대기 시간(분) |
| `pallet_wrap_time_min` | 팔레트 포장 소요 시간(분) |
| `cross_dock_ratio` | 크로스 도킹(입고 즉시 출고) 비율 |
| `sort_accuracy_pct` | 분류 정확도 (%) |
| `label_print_queue` | 라벨 인쇄 대기열 길이 |
| `quality_check_rate` | 품질 검사 통과율 |
| `dock_to_stock_hours` | 도크 입고 후 재고 등록까지 소요 시간(시간) |
| `shift_handover_delay_min` | 교대 인수인계 지연 시간(분) |

### 3-6. 환경·설비 피처 (Environment & Facility)

| 컬럼 | 설명 |
|------|------|
| `warehouse_temp_avg` | 창고 내부 평균 온도 (°C) |
| `humidity_pct` | 창고 내부 습도 (%) |
| `cold_storage_temp_c` | 냉장 보관 구역 온도 (°C) |
| `zone_temp_variance` | 구역 간 온도 분산 |
| `external_temp_c` | 외부 온도 (°C) |
| `wind_speed_kmh` | 외부 풍속 (km/h) |
| `precipitation_mm` | 강수량 (mm) |
| `lighting_level_lux` | 조명 밝기 (lux) |
| `lighting_zone_variance` | 구역 간 조명 분산 |
| `ambient_noise_db` | 주변 소음 (dB) |
| `floor_vibration_idx` | 바닥 진동 지수 |
| `air_quality_idx` | 공기질 지수 |
| `co2_level_ppm` | CO₂ 농도 (ppm) |
| `hvac_power_kw` | HVAC(공조) 전력 소비 (kW) |
| `ups_battery_pct` | UPS(무정전 전원장치) 배터리 잔량 (%) |

### 3-7. 인력·IT 피처 (Workforce & IT)

| 컬럼 | 설명 |
|------|------|
| `staff_on_floor` | 현장 근무 직원 수 |
| `forklift_active_count` | 활성 지게차 수 |
| `worker_avg_tenure_months` | 직원 평균 근속 개월 수 |
| `safety_score_monthly` | 월간 안전 점수 |
| `wms_response_time_ms` | WMS(창고 관리 시스템) 응답 시간 (ms) |
| `scanner_error_rate` | 바코드 스캐너 오류율 |
| `barcode_read_success_rate` | 바코드 인식 성공률 |
| `wifi_signal_db` | WiFi 신호 강도 (dB) |
| `network_latency_ms` | 네트워크 지연 시간 (ms) |
| `shift_hour` | 현재 교대 내 경과 시간 |
| `day_of_week` | 요일 (0=월요일) |

### 3-8. 재고·KPI 피처 (Inventory & KPI)

| 컬럼 | 설명 |
|------|------|
| `storage_density_pct` | 보관 밀도 (%) |
| `vertical_utilization` | 수직 공간 활용률 |
| `racking_height_avg_m` | 평균 래킹(선반) 높이 (m) |
| `inventory_turnover_rate` | 재고 회전율 |
| `backorder_ratio` | 백오더(재고 부족 주문) 비율 |
| `avg_package_weight_kg` | 평균 패키지 무게 (kg) |
| `packaging_material_cost` | 포장 재료 비용 |
| `kpi_otd_pct` | KPI: 정시 배송률 (%) |
| `maintenance_schedule_score` | 유지보수 일정 준수 점수 |
| `conveyor_speed_mps` | 컨베이어 속도 (m/s) |

### 3-9. 타깃 컬럼

| 컬럼 | 설명 |
|------|------|
| `avg_delay_minutes_next_30m` | **예측 대상** — 현재 타임슬롯 기준 향후 30분간 평균 출고 지연 시간(분) |

## 4. layout_info.csv 컬럼 정의

| 컬럼 | 설명 |
|------|------|
| `layout_id` | 창고 레이아웃 ID (train/test와 조인 키) |
| `layout_type` | 레이아웃 유형 (`narrow` / `grid` / `wide` 등) |
| `aisle_width_avg` | 평균 통로 너비 (m) |
| `intersection_count` | 교차로(로봇 경로 분기점) 수 |
| `one_way_ratio` | 일방통행 통로 비율 |
| `pack_station_count` | 패킹 스테이션 수 |
| `charger_count` | 로봇 충전기 수 |
| `layout_compactness` | 레이아웃 압축도 (높을수록 공간 효율적) |
| `zone_dispersion` | 구역 분산도 (구역이 얼마나 퍼져 있는지) |
| `robot_total` | 해당 창고 총 로봇 수 |
| `building_age_years` | 건물 연령 (년) |
| `floor_area_sqm` | 바닥 면적 (m²) |
| `ceiling_height_m` | 천장 높이 (m) |
| `fire_sprinkler_count` | 화재 스프링클러 수 |
| `emergency_exit_count` | 비상구 수 |

## 5. 솔루션 개요

### 모델 구조
```
[ Level-1 ]
  LightGBM (15-Fold GroupKFold)  → OOF predictions
  CatBoost (15-Fold GroupKFold)  → OOF predictions

[ Level-2 ]
  LightGBM Meta-model
  Input: Level-1 OOF preds + Top-100 original features
  → Final prediction
```

### 핵심 기법
- **log1p transform**: 타깃에 log1p 적용 후 학습, 예측 시 expm1 역변환 (MAE 개선)
- **GroupKFold**: scenario_id 단위로 분할하여 데이터 누수 방지
- **Sample Weighting**: 고지연(≥40분) 샘플에 높은 가중치 + 시나리오 단위 multiplier
- **Scenario Features**: 25개 슬롯 전체 제공 특성을 활용한 시나리오 집계/궤적/상대 피처
- **Lead Features**: 미래 슬롯 값 (정보 누수 아님 — 25슬롯 동시 제공)